<a href="https://colab.research.google.com/github/assafshabat/PINNs-research/blob/main/MIM_and_understanding_and_mitigating_gradient_pathologies_in_PINNs_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Biharmonic equation on $\Omega = [0,1]^{d}$***

### Import libraries, set seeds & device

In [ ]:
''' cell 1 '''
# import libraries
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

import time

# Set random seeds
np.random.seed(1)
torch.manual_seed(1)

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"The program is running on: {device}")

The program is running on: cpu


### Build PINN Architecture

# Activation function, ResNet block and models

In [ ]:
''' cell 2 '''

class SineActivation(nn.Module): # They used Tanh
    def __init__(self, omega_0=1.0):
        super().__init__()
        self.omega_0 = omega_0

    def forward(self, x):
        return torch.sin(self.omega_0 * x)

class ModifiedMLP_MIM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_blocks):
        super().__init__()
        # self.act = nn.Tanh()
        self.act = SineActivation(omega_0=1.0)
        self.encoder_U = nn.Linear(input_dim, hidden_dim)
        self.encoder_V = nn.Linear(input_dim, hidden_dim)
        self.first_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(num_blocks - 1)])
        self.out_layer = nn.Linear(hidden_dim, 2) # MIM_p^1 outputs 2 values (u, q)
        self._init_weights()

    def _init_weights(self):
            with torch.no_grad():
                for name, m in self.named_modules():
                    if isinstance(m, nn.Linear):
                        num_input = m.weight.size(-1)

                        if 'encoder' in name or 'first_layer' in name:
                            bound = 1.0 / num_input  # initialization for the first layers
                        else:
                            bound = np.sqrt(6.0 / num_input)  # initialization for the hidden layers

                        m.weight.uniform_(-bound, bound)
                        if m.bias is not None:
                            m.bias.zero_()

    def forward(self, x):
        U = self.act(self.encoder_U(x)) # Φ(XW^{1}+b^{1})
        V = self.act(self.encoder_V(x)) # Φ(XW^{2}+b^{2})
        H = self.act(self.first_layer(x)) # Φ(XW^{z,1}+b^{z,1})

        for layer in self.hidden_layers:
            Z = self.act(layer(H)) # Φ(H^{(k)}W^{z,k}+b^{z,k})
            H = (1.0 - Z) * U + Z * V  # H^{(k+1)}

        out = self.out_layer(H) # f(x)
        u = out[:, 0:1]
        q = out[:, 1:2]
        return u, q

# Choose the model and optimizer

In [ ]:
''' cell 3 '''

# Hyper-parameters
hidden_dims = [10, 20, 30, 40, 50] # Width
num_blocks = 2 # Depth
learning_rate = 1e-3 # Learning rate

# Training & sampling configuration
dimensions = list(range(2, 8))  # 2 ≤ d ≤ 7
num_iters = 5000                # Number of iterations
n_interior = 512                # Number of interior points N in [0,1]^d
n_boundary_per_face = 32        # Number of boundary points per face (2*d*n_boundary_per_face)
n_test = 10000                  # Number of test points

### Set the function $f$ in the Biharmonic and exact solution:
### $\Delta^{2}u = f$

In [ ]:
''' cell 4 '''

def exact_solution_1(x): # u(x) = (1/(d-1))*sum_{i=1}^{d-1}sin(2*x_{i}+1)*cos(2*x_{i+1}+1)
    d = x.shape[1] # Dimension (x_{1},...,x_{d})
    u = torch.zeros((x.shape[0], 1), device=x.device, dtype=x.dtype) # Create 0 tensor of size Batch×1
    for i in range(d - 1):
        u = u + torch.sin(2.0 * x[:, i:i+1] + 1.0) * torch.cos(2.0 * x[:, i+1:i+2] + 1.0) # Sum
    u = u / (d - 1)
    return u

def forcing_function_1(x): # Δu_{i} = -8*u_{i}, so Δ^2(u) = 64 * u
    f = 64.0 * exact_solution_1(x)
    return f

def exact_solution_2(x): # u(x) = log((x_{1}+2)^2+(x_{2}+1)^2)
    u = torch.log((x[:, 0:1] + 2.0)**2 + (x[:, 1:2] + 1.0)**2)
    return u

def forcing_function_2(x): # Δ^2(u) = 0
    f = torch.zeros((x.shape[0], 1), device=x.device, dtype=x.dtype)
    return f

def exact_solution_3(x): # u(x) = prod_{i=1}^d 16*exp(x_{i}-0.5)*x_{i}^{2}*(1-x_{i})^{2}
    q_val = 16.0 * torch.exp(x - 0.5) * (x**2) * ((1.0 - x)**2) # q(x)
    u = torch.prod(q_val, dim=1, keepdim=True) # product of q(x)
    return u

def forcing_function_3(x):
    d = x.shape[1] # Dimension (x_{1},...,x_{d})
    f = torch.zeros((x.shape[0], 1), device=x.device, dtype=x.dtype) # Create 0 tensor of size Batch×1

    q_val = 16.0 * torch.exp(x - 0.5) * (x**4 - 2.0*x**3 + x**2) # q(x)
    q_2 = 16.0 * torch.exp(x - 0.5) * (x**4 + 6.0*x**3 + x**2 - 8.0*x + 2.0) # q^{(2)}(x)
    q_4 = 16.0 * torch.exp(x - 0.5) * (x**4 + 14.0*x**3 + 49.0*x**2 + 32.0*x - 12.0) # q^{(4)}(x))

    # Δ^2(u) = sum_{i}^{d} (d^{4}u/dx_{i}^{4}) + 2*(d^{4}u/(dx_{i}^{2}dx_{j}^{2}))

    # d^{4}u/dx_{i}^{4}
    for i in range(d):
        term = q_4[:, i:i+1]
        for k in range(d):
            if k != i:
                term = term * q_val[:, k:k+1]
        f = f + term

    # 2*(d^{4}u/(dx_{i}^{2}dx_{j}^{2}))
    for i in range(d):
        for j in range(i + 1, d):
            term = q_2[:, i:i+1] * q_2[:, j:j+1]
            for k in range(d):
                if k != i and k != j:
                    term = term * q_val[:, k:k+1]
            f = f + 2.0 * term

    return f

### Calculate $u=g_{1}$ and $\frac{\partial u}{\partial n}=g_{2}$ on $\partial\Omega$

In [ ]:
''' cell 5 '''

def boundary_g1(x_bnd, exact_sol_fn): # Computes exact Dirichlet boundary target g1 = u(x) on ∂Ω
    g1 = exact_sol_fn(x_bnd).detach()
    return g1


def boundary_g2(x_bnd, normals_bnd, exact_sol_fn): # Computes exact ∇(u) and Neumann boundary target g2 = du/dn = ∇(u)*n on ∂Ω
    x_tmp = x_bnd.detach().clone().requires_grad_(True)
    u_ex = exact_sol_fn(x_tmp)

    grad_exact = torch.autograd.grad(
        outputs=u_ex, inputs=x_tmp,
        grad_outputs=torch.ones_like(u_ex),
        create_graph=False
    )[0].detach()  # Create ∇(u) tensor of size Batch×d

    du_dn_exact = torch.sum(grad_exact * normals_bnd, dim=1, keepdim=True) # Create du/dn tensor of size Batch×1
    return du_dn_exact, grad_exact

### Calculate gradient, divergence and Laplacian

In [ ]:
''' cell 6 '''

def compute_scalar_gradient(scalar_field, x): # Computes the d-dimensional gradient: ∇(scalar_field) in R^{N x d}
    grad_scalar_field = torch.autograd.grad(
        outputs=scalar_field, inputs=x,
        grad_outputs=torch.ones_like(scalar_field),
        create_graph=True
    )[0] # Create ∇(scalar_field) tensor of size Batch×d
    return grad_scalar_field


def compute_vector_divergence(vec_field, x): # Computes the d-dimensional divergence: ∇·(vec_field) = sum_{i=1}^d d(vec_field_i)/dx_i in R^{N x 1}
    div = torch.zeros((x.shape[0], 1), device=x.device, dtype=x.dtype) # Create 0 tensor of size Batch×1
    d = x.shape[1] # Dimension (x_{1},...,x_{d})
    for i in range(d):
        v_i = vec_field[:, i:i+1]
        dv_i_dxi = torch.autograd.grad(
            outputs=v_i, inputs=x,
            grad_outputs=torch.ones_like(v_i),
            create_graph=True
        )[0][:, i:i+1]
        div = div + dv_i_dxi
    return div # Tensor of size Batch×1

def compute_scalar_laplacian(scalar_field, x): # Computes the d-dimensional Laplacian: ∇·(∇(scalar_field))
    grad_scalar_field = compute_scalar_gradient(scalar_field, x) # ∇(scalar_field)
    divergence_scalar_field = compute_vector_divergence(grad_scalar_field, x) # ∇·(∇(scalar_field))
    return divergence_scalar_field # Tensor of size Batch×1

### Compute derivatives of $MIM_{p}$.

In [ ]:
''' cell 7 '''

def compute_mim_p_derivatives(model, x): # MIM_p^1 and MIM_p^2: returns u, q, Laplacian(u), Laplacian(q)
    u, q = model(x) # Gives u,q from model
    lap_u = compute_scalar_laplacian(u, x) # Calculate ∇·(∇(u))
    lap_q = compute_scalar_laplacian(q, x) # Calculate ∇·(∇(q))
    return u, q, lap_u, lap_q


# def compute_mim_a_derivatives(model, x): # MIM_a^1 and MIM_a^2: returns u, p, q, w, grad(u), div(p), grad(q), div(w)
#     u, p, q, w = model(x) # Gives u,p,q,w from model
#     grad_u = compute_scalar_gradient(u, x) # Calculate ∇(u)
#     div_p = compute_vector_divergence(p, x) # Calculate Δ(u)
#     grad_q = compute_scalar_gradient(q, x) # Calculate ∇(Δ(u))
#     div_w = compute_vector_divergence(w, x) # Calculate Δ^{2}(u)
#     return u, p, q, w, grad_u, div_p, grad_q, div_w

### Detects whether the model is a $MIM_{p}$ or $MIM_{a}$

In [ ]:
''' cell 8 '''

def compute_mim_loss(model, x_int, x_bnd, normals_bnd, exact_sol_fn, forcing_fn):
    f_int = forcing_fn(x_int).detach() # Caculate f for interior points
    g1_bnd = boundary_g1(x_bnd, exact_sol_fn) # Calculate g_{1}
    g2_bnd, _ = boundary_g2(x_bnd, normals_bnd, exact_sol_fn) # Calculate g_{2}

    if isinstance(model, (ModifiedMLP_MIM)): # Check if model is MIM_p (2 outputs) or MIM_a (4 outputs) (for now the only model in th dictionary is ModifiedMLP_MIM)

        u_int, q_int, lap_u, lap_q = compute_mim_p_derivatives(model, x_int)
        loss_pde = torch.mean((q_int - lap_u)**2) + torch.mean((lap_q - f_int)**2) # Interior residuals: ||q-Δu||^{2}+||Δq-f||^{2}

        u_bnd, _ = model(x_bnd)
        grad_u_bnd = compute_scalar_gradient(u_bnd, x_bnd)
        du_dn_pred = torch.sum(grad_u_bnd * normals_bnd, dim=1, keepdim=True)

        loss_bc1 = torch.mean((u_bnd - g1_bnd)**2) # Boundary residuals 1: ||u-g_{1}||^{2}
        loss_bc2 = torch.mean((du_dn_pred - g2_bnd)**2) # Boundary residuals 2: ||du/dn-g_{2}||^{2}

    # else:
    #     # Interior residuals: use torch.mean on vector fields so loss doesn't scale up with d
    #     u_int, p_int, q_int, w_int, grad_u, div_p, grad_q, div_w = compute_mim_a_derivatives(model, x_int)
    #     loss_pde = (torch.mean((p_int - grad_u)**2) + torch.mean((q_int - div_p)**2) + torch.mean((w_int - grad_q)**2) + torch.mean((div_w - f_int)**2))

    #     u_bnd, p_bnd, _, _ = model(x_bnd)
    #     du_dn_pred = torch.sum(p_bnd * normals_bnd, dim=1, keepdim=True)

    #     loss_bc1 = torch.mean((u_bnd - g1_bnd)**2) # Boundary residuals 1: ||u-g_{1}||^{2}
    #     loss_bc2 = torch.mean((du_dn_pred - g2_bnd)**2) # Boundary residuals 2: ||p*n-g_{2}||^{2}

    return loss_pde, loss_bc1, loss_bc2

### Sample points and normals

In [ ]:
''' cell 9 '''

def sample_points_nd(n_interior, n_boundary_per_face, d, device): # Samples interior points in [0,1]^d and boundary points across all 2*d faces.

    x_int = torch.rand(n_interior, d, device=device, requires_grad=True) # Random tensor (x_{1},...,x_{d}), (interior point)

    bnd_points_list = []
    normals_list = []

    for i in range(d):
        # Face x_{i} = 0 (outward normal: -e_i)
        p_zero = torch.rand(n_boundary_per_face, d, device=device) # Random tensor (x_{1},...,x_{d})
        p_zero[:, i] = 0.0 # Random tensor (x_{1},...,x_{d}) ), (boundary point on Face x_{i} = 0)
        n_zero = torch.zeros(n_boundary_per_face, d, device=device) # 0 tensor (x_{1},...,x_{d})
        n_zero[:, i] = -1.0 # Normal tensor (0,...,0,-1,0,...0), (normal)

        # Face x_i = 1 (outward normal: +e_i)
        p_one = torch.rand(n_boundary_per_face, d, device=device) # Random tensor (x_{1},...,x_{d})
        p_one[:, i] = 1.0 # Random tensor (x_{1},...,x_{d}), (boundary point on Face x_{i} = 1)
        n_one = torch.zeros(n_boundary_per_face, d, device=device) # 0 tensor (x_{1},...,x_{d})
        n_one[:, i] = 1.0 # Normal tensor (0,...,0,-1,0,...0), (normal)

        bnd_points_list.extend([p_zero, p_one]) # Add points
        normals_list.extend([n_zero, n_one]) # Add normals

    x_bnd = torch.cat(bnd_points_list, dim=0).requires_grad_(True) # Concatenated boundary points
    normals_bnd = torch.cat(normals_list, dim=0) # Concatenated normals

    return x_int, x_bnd, normals_bnd

# Meta parameters $\lambda_{1}$ and $\lambda_{2}$

In [ ]:
''' cell 10 '''

def update_lambda_weights(model, loss_pde, loss_bc1, loss_bc2, lambda1, lambda2, alpha_lra, eps, min_lambda=1.0, max_lambda=10.0):
    params = [p for p in model.parameters() if p.requires_grad] # List of the parameters

    grads_pde = torch.autograd.grad(loss_pde, params, retain_graph=True, allow_unused=True) # Compute parameter gradients for L_pde
    grads_bc1 = torch.autograd.grad(loss_bc1, params, retain_graph=True, allow_unused=True) # Compute parameter gradients for L_bc1
    grads_bc2 = torch.autograd.grad(loss_bc2, params, retain_graph=True, allow_unused=True) # Compute parameter gradients for L_bc2

    flat_pde = torch.cat([g.reshape(-1).abs() for g in grads_pde if g is not None]) # Takes each gradients from grads_pde, reshape and concatenate into a vector
    flat_bc1 = torch.cat([g.reshape(-1).abs() for g in grads_bc1 if g is not None]) # Takes each gradients from grads_bc1, reshape and concatenate into a vector
    flat_bc2 = torch.cat([g.reshape(-1).abs() for g in grads_bc2 if g is not None]) # Takes each gradients from grads_bc2, reshape and concatenate into a vector

    with torch.no_grad():
        max_grad_pde = torch.max(flat_pde).item()   # max |∇_{θ}L_{pde}|
        mean_grad_bc1 = torch.mean(flat_bc1).item() # mean |∇_{θ}L_{bc1}|
        mean_grad_bc2 = torch.mean(flat_bc2).item() # mean |∇_{θ}L_{bc2}|

        # Clip instantaneous ratios to [1.0, 1000.0]
        lambda1_hat = float(np.clip(max_grad_pde / (mean_grad_bc1 + eps), min_lambda, max_lambda)) # λ_{1} hat
        lambda2_hat = float(np.clip(max_grad_pde / (mean_grad_bc2 + eps), min_lambda, max_lambda)) # λ_{2} hat

        # Moving average update clamped to [1.0, 1000.0]
        lambda1 = float(np.clip((1.0 - alpha_lra) * lambda1 + alpha_lra * lambda1_hat, min_lambda, max_lambda)) # Update λ_{1}
        lambda2 = float(np.clip((1.0 - alpha_lra) * lambda2 + alpha_lra * lambda2_hat, min_lambda, max_lambda)) # Update λ_{2}

    return lambda1, lambda2

# Dictionarys for the equations and models

In [ ]:
''' cell 11 '''

# Dictionary of the 3 Biharmonic equations
equations = {
    "Eq1_Trig": (exact_solution_1, forcing_function_1),
    "Eq2_Log":  (exact_solution_2, forcing_function_2),
    "Eq3_Prod": (exact_solution_3, forcing_function_3),
}

# Dictionary of the 4 MIM architectures
architectures = {
    "ModifiedMLP_MIM": ModifiedMLP_MIM,
}

# Count learnable parameters

In [ ]:
''' cell 12 '''

def count_active_parameters(model, d, hidden_dim): # Counts trainable parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # ModifiedMLP_MIM does not use zero-padding, so all parameters are active
    if isinstance(model, ModifiedMLP_MIM) or d >= hidden_dim:
        return total_params

    # # Only subtract dead zero-padded weights for the ResNet MIM architectures
    # dead_per_trunk = (hidden_dim - d) * hidden_dim
    # arch_name = type(model).__name__
    # if arch_name in ["MIMNet_p1", "MIMNet_a1"]:
    #     num_trunks = 1
    # elif arch_name == "MIMNet_p2":
    #     num_trunks = 2
    # elif arch_name == "MIMNet_a2":
    #     num_trunks = 4
    # else:
    #     num_trunks = 0

    # return total_params - num_trunks * dead_per_trunk

# Training

In [ ]:
''' cell 13 '''

# Meta-parameter hyper-parameters (Algorithm 1)
alpha_lra = 0.1   # Moving average weight on new \hat{\lambda}_i
eps = 1e-12
update_freq = 10  # Updates lambda_i every 10 iterations

results = []
benchmark_start_time = time.perf_counter()

for eq_name, (exact_sol_fn, forcing_fn) in equations.items():
    print(f"\n==================== {eq_name} ====================")
    eq_start_time = time.perf_counter()

    for d in dimensions:
        print(f"--- Dimension d = {d} ---")
        # Generate a fixed test set in [0,1]^d for fair comparison across widths and architectures
        torch.manual_seed(42)
        x_test = torch.rand(n_test, d, device=device)
        with torch.no_grad():
            u_exact_test = exact_sol_fn(x_test)

        for arch_name, ModelClass in architectures.items():
            for h_dim in hidden_dims:
                # Reset seeds so every run starts from a reproducible state
                np.random.seed(1)
                torch.manual_seed(1)

                # Initialize model with current h_dim and optimizer
                model = ModelClass(input_dim=d, hidden_dim=h_dim, num_blocks=num_blocks).to(device)
                num_params = count_active_parameters(model, d, h_dim)
                optimizer = optim.Adam(model.parameters(), lr=learning_rate)

                # Reset meta-parameters and scheduler for each training run
                lambda1 = 1.0
                lambda2 = 1.0
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_iters, eta_min=1e-5)

                # Start timer
                if device.type == "cuda":
                    torch.cuda.synchronize()
                train_start_time = time.perf_counter()

                # Training loop
                model.train()
                for it in range(num_iters):
                    optimizer.zero_grad()

                    x_int, x_bnd, normals_bnd = sample_points_nd(
                        n_interior, n_boundary_per_face, d, device
                    )

                    # 1. Compute unweighted loss components
                    loss_pde, loss_bc1, loss_bc2 = compute_mim_loss(
                        model, x_int, x_bnd, normals_bnd, exact_sol_fn, forcing_fn
                    )

                    # 2. Update meta-parameters lambda1, lambda2 every update_freq iterations
                    if it % update_freq == 0:
                        lambda1, lambda2 = update_lambda_weights(
                            model, loss_pde, loss_bc1, loss_bc2,
                            lambda1, lambda2, alpha_lra=alpha_lra, eps=eps
                        )

                    # 3. Compute weighted composite loss and update weights
                    loss = loss_pde + lambda1 * loss_bc1 + lambda2 * loss_bc2
                    loss.backward()
                    optimizer.step()
                    scheduler.step()

                # Stop timer
                if device.type == "cuda":
                    torch.cuda.synchronize()
                train_time_s = time.perf_counter() - train_start_time
                time_per_iter_s = train_time_s / num_iters

                # Evaluate L^2 error on test set
                model.eval()
                with torch.no_grad():
                    u_pred_test = model(x_test)[0]
                    l2_error = torch.sqrt(torch.mean((u_pred_test - u_exact_test) ** 2)).item()

                print(
                    f"  {arch_name:15s} | h_dim: {h_dim:2d} | Params: {num_params:4d} | "
                    f"Time: {train_time_s:6.2f}s ({time_per_iter_s:.4f}s/it) | "
                    f"L2(u): {l2_error:.4e} | λ1: {lambda1:.1f}, λ2: {lambda2:.1f}"
                )

                results.append({
                    "Equation": eq_name,
                    "Dimension": d,
                    "Architecture": arch_name,
                    "Hidden_Dim": h_dim,
                    "Params": num_params,
                    "L2_Error_u": l2_error,
                    "Train_Time_s": train_time_s,
                    "Time_Per_Iter_s": time_per_iter_s,
                    "Final_Loss": loss.item(),
                    "Final_Lambda1": lambda1,
                    "Final_Lambda2": lambda2,
                })

    eq_total_time = time.perf_counter() - eq_start_time
    print(f"--> Completed {eq_name} in {eq_total_time:.2f}s ({eq_total_time/60:.2f} min)")

total_benchmark_time = time.perf_counter() - benchmark_start_time
print(f"\n====================================================")
print(f"All {len(results)} training runs finished in {total_benchmark_time:.2f}s ({total_benchmark_time/60:.2f} min)")
print(f"====================================================")

df_results = pd.DataFrame(results)
df_results.to_csv("mim_biharmonic_benchmark.csv", index=False)


==================== Eq1_Trig ====================
--- Dimension d = 2 ---
  ModifiedMLP_MIM | h_dim: 10 | Params:  222 | Time: 129.70s (0.0259s/it) | L2(u): 1.1329e-01 | λ1: 1.1, λ2: 1.1
  ModifiedMLP_MIM | h_dim: 20 | Params:  642 | Time: 205.37s (0.0411s/it) | L2(u): 6.8264e-02 | λ1: 1.1, λ2: 1.1
  ModifiedMLP_MIM | h_dim: 30 | Params: 1262 | Time: 280.46s (0.0561s/it) | L2(u): 1.7756e-02 | λ1: 1.1, λ2: 1.1
  ModifiedMLP_MIM | h_dim: 40 | Params: 2082 | Time: 365.67s (0.0731s/it) | L2(u): 1.8914e-02 | λ1: 1.1, λ2: 1.1
  ModifiedMLP_MIM | h_dim: 50 | Params: 3102 | Time: 462.69s (0.0925s/it) | L2(u): 1.6410e-02 | λ1: 1.1, λ2: 1.1
--> Completed Eq1_Trig in 1443.93s (24.07 min)

==================== Eq2_Log ====================
--- Dimension d = 2 ---
  ModifiedMLP_MIM | h_dim: 10 | Params:  222 | Time: 128.69s (0.0257s/it) | L2(u): 5.9697e-03 | λ1: 1.1, λ2: 1.1
  ModifiedMLP_MIM | h_dim: 20 | Params:  642 | Time: 204.93s (0.0410s/it) | L2(u): 3.2014e-03 | λ1: 1.1, λ2: 1.1
  ModifiedM

### Plot

In [ ]:
''' cell 14 '''

# Load from CSV if df_results is not currently in memory
if "df_results" not in locals():
    df_results = pd.read_csv("mim_biharmonic_benchmark.csv")

eq_Metadata = [
    ("Eq1_Trig", "Equation 1: Coupled Trigonometric"),
    ("Eq2_Log",  "Equation 2: Logarithmic"),
    ("Eq3_Prod", "Equation 3: Product Polynomial-Exponential"),
]

dimensions = sorted(df_results["Dimension"].unique())
arch_list = df_results["Architecture"].unique()
cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(dimensions)))
markers = ["o", "s", "^", "D", "v", "P"]


def plot_loglog_across_hidden_dims(df, y_col, y_label, title_prefix, filename_prefix):
    for arch_name in arch_list:
        df_arch = df[df["Architecture"] == arch_name]

        fig, axes = plt.subplots(1, 3, figsize=(19, 5.5), dpi=120)
        fig.suptitle(
            f"{title_prefix} — {arch_name} (hidden_dim $\in$ {{10, 20, 30, 40, 50}})",
            fontsize=15,
            fontweight="bold",
            y=1.03,
        )

        for idx, (eq_key, eq_title) in enumerate(eq_Metadata):
            ax = axes[idx]
            df_eq = df_arch[df_arch["Equation"] == eq_key]

            for d_idx, d in enumerate(dimensions):
                # Sort by Hidden_Dim (10 -> 20 -> 30 -> 40 -> 50) so Params strictly increase
                df_d = df_eq[df_eq["Dimension"] == d].sort_values(by="Hidden_Dim")

                ax.loglog(
                    df_d["Params"],
                    df_d[y_col],
                    marker=markers[d_idx % len(markers)],
                    color=cmap[d_idx],
                    linewidth=1.8,
                    markersize=7,
                    label=f"d = {d}",
                )

                # Annotate the hidden_dim values (10 and 50) at the ends of each curve
                for h_val in [df_d["Hidden_Dim"].min(), df_d["Hidden_Dim"].max()]:
                    row = df_d[df_d["Hidden_Dim"] == h_val].iloc[0]
                    ax.annotate(
                        f"h={int(h_val)}",
                        (row["Params"], row[y_col]),
                        textcoords="offset points",
                        xytext=(4, 3),
                        fontsize=7.5,
                        color=cmap[d_idx],
                    )

            ax.set_title(eq_title, fontsize=12, fontweight="bold")
            ax.set_xlabel("Number of Learnable Parameters (log scale)", fontsize=11)
            if idx == 0:
                ax.set_ylabel(y_label, fontsize=11)
            ax.grid(True, which="both", linestyle=":", linewidth=0.7, alpha=0.7)
            ax.legend(title="Dimension", fontsize=9.5, title_fontsize=10)

        plt.tight_layout()
        plt.savefig(f"{filename_prefix}_{arch_name}.png", bbox_inches="tight")
        plt.show()


# 1. Plot L^2 Error vs. Number of Learnable Parameters
plot_loglog_across_hidden_dims(
    df=df_results,
    y_col="L2_Error_u",
    y_label="$L^2(\Omega)$ Error of $u$ (log scale)",
    title_prefix="Log-Log: $L^2(\Omega)$ Error vs. Learnable Parameters",
    filename_prefix="loglog_l2_error_vs_params",
)

# 2. Plot Training Running Time vs. Number of Learnable Parameters
plot_loglog_across_hidden_dims(
    df=df_results,
    y_col="Train_Time_s",
    y_label="Training Running Time in Seconds (log scale)",
    title_prefix="Log-Log: Training Running Time vs. Learnable Parameters",
    filename_prefix="loglog_runtime_vs_params",
)

<>:25: SyntaxWarning: invalid escape sequence '\i'
<>:77: SyntaxWarning: invalid escape sequence '\O'
<>:78: SyntaxWarning: invalid escape sequence '\O'
<>:25: SyntaxWarning: invalid escape sequence '\i'
<>:77: SyntaxWarning: invalid escape sequence '\O'
<>:78: SyntaxWarning: invalid escape sequence '\O'
/tmp/ipykernel_926/3777573007.py:25: SyntaxWarning: invalid escape sequence '\i'
  f"{title_prefix} — {arch_name} (hidden_dim $\in$ {{10, 20, 30, 40, 50}})",
/tmp/ipykernel_926/3777573007.py:77: SyntaxWarning: invalid escape sequence '\O'
  y_label="$L^2(\Omega)$ Error of $u$ (log scale)",
/tmp/ipykernel_926/3777573007.py:78: SyntaxWarning: invalid escape sequence '\O'
  title_prefix="Log-Log: $L^2(\Omega)$ Error vs. Learnable Parameters",


NameError: name 'pd' is not defined